# Modelo de Regresión Logística

Este notebook implementa la Regresión Logística como modelo supervisado base para la detección y clasificación de tráfico anómalo en el dataset LITNET-2020.

Se desarrollan dos tareas:

1. Clasificación binaria: distinguir entre tráfico normal (`attack_a = 0`) y tráfico de ataque (`attack_a = 1`).
2. Clasificación multiclase: identificar el tipo específico de ataque utilizando únicamente los registros con `attack_a = 1` y la variable objetivo `attack_t`.

El notebook reutiliza el particionamiento y las funciones de preprocesamiento definidas en `03_preprocesamiento_para_modelado.ipynb`. El preprocesamiento se ajustará exclusivamente con datos de entrenamiento (`fit`) y posteriormente se aplicará sin reajuste sobre validación y prueba (`transform`).

Durante la fase de desarrollo se utilizará `dataset_modelado_dev.parquet` para reducir el costo computacional. La configuración seleccionada posteriormente podrá evaluarse sobre el conjunto completo `dataset_modelado_con_split.parquet`.

El conjunto de prueba se reserva para la evaluación final y no se utilizará durante la selección de estrategias de desbalance, hiperparámetros o umbrales de clasificación.

In [1]:
from pathlib import Path
import sys
import json
import random
import warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
RANDOM_STATE = 42
SECOND_RANDOM_STATE = 2026
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

In [2]:
PROJECT_ROOT = Path("..").resolve()
DATA_PATH = PROJECT_ROOT / "data"
INTERIM_PATH = DATA_PATH / "interim"
RESULTS_PATH = PROJECT_ROOT / "results"
PREPROCESSING_RESULTS_PATH = RESULTS_PATH / "preprocessing"
LR_RESULTS_PATH = (RESULTS_PATH / "modeling" / "logistic_regression")
LR_BINARY_RESULTS_PATH = (LR_RESULTS_PATH / "binary")
LR_MULTICLASS_RESULTS_PATH = (LR_RESULTS_PATH / "multiclass")
MODELS_PATH = PROJECT_ROOT / "models"
LR_MODELS_PATH = (MODELS_PATH / "logistic_regression")
for path in [LR_RESULTS_PATH, LR_BINARY_RESULTS_PATH, LR_MULTICLASS_RESULTS_PATH, LR_MODELS_PATH]:
    path.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:")
print(PROJECT_ROOT)
print("\nResultados Regresión Logística:")
print(LR_RESULTS_PATH)
print("\nModelos Regresión Logística:")
print(LR_MODELS_PATH)

PROJECT_ROOT:
C:\Users\Laura\Documents\TrabajoGrado2026\TG2026

Resultados Regresión Logística:
C:\Users\Laura\Documents\TrabajoGrado2026\TG2026\results\modeling\logistic_regression

Modelos Regresión Logística:
C:\Users\Laura\Documents\TrabajoGrado2026\TG2026\models\logistic_regression


In [3]:
# Modo de ejecución - muestra DEV o dataset completo
USE_DEV_SAMPLE = True
DATASET_MODE = (
    "development"
    if USE_DEV_SAMPLE
    else "full"
)
print(f"Modo de ejecución: {DATASET_MODE}")
if USE_DEV_SAMPLE:
    print("Se utilizará dataset_modelado_dev.parquet.")
else:
    print("Se utilizará dataset_modelado_con_split.parquet.")

Modo de ejecución: development
Se utilizará dataset_modelado_dev.parquet.


### Configuraciones de preprocesamiento y desbalance

Las decisiones sobre variables, transformaciones y estrategias de desbalance no se redefinen en este notebook. Se cargan desde los archivos generados previamente, garantizando que los distintos modelos utilicen una configuración común.

In [4]:
# Archivos de configuración
PREPROCESSING_CONFIG_PATH = (PREPROCESSING_RESULTS_PATH / "preprocessing_config.json")
IMBALANCE_CONFIG_PATH = (PREPROCESSING_RESULTS_PATH / "imbalance_config.json")
for config_path in [PREPROCESSING_CONFIG_PATH, IMBALANCE_CONFIG_PATH,]:
    if not config_path.exists():
        raise FileNotFoundError(
            f"No se encontró el archivo: {config_path}"
        )

with open(PREPROCESSING_CONFIG_PATH, "r", encoding="utf-8") as file:
    preprocessing_config = json.load(file)

with open(IMBALANCE_CONFIG_PATH, "r", encoding="utf-8") as file:
    imbalance_config = json.load(file)

print("Configuraciones cargadas correctamente.")
print("\nNúmero de variables predictoras:", len(preprocessing_config["predictor_candidate_columns"]))
print("\nEstrategias de desbalance:")
print(imbalance_config["imbalance_strategies"])

Configuraciones cargadas correctamente.

Número de variables predictoras: 22

Estrategias de desbalance:
['none', 'class_weight', 'smote', 'adasyn', 'smote_enn']


In [5]:
# Validación básica de configuraciones
required_preprocessing_keys = {
    "numeric_log_columns",
    "binary_flag_columns",
    "one_hot_columns",
    "frequency_encoding_columns",
    "ip_columns",
    "port_feature_columns",
    "target_columns",
    "predictor_candidate_columns",
    "binary_target_column",
    "multiclass_target_column",
}
missing_preprocessing_keys = (required_preprocessing_keys - set(preprocessing_config))
if missing_preprocessing_keys:
    raise ValueError("Faltan elementos en preprocessing_config: "f"{sorted(missing_preprocessing_keys)}")

if "imbalance_strategies" not in imbalance_config:
    raise ValueError("imbalance_config no contiene'imbalance_strategies'.")

print("Configuraciones validadas correctamente.")

Configuraciones validadas correctamente.


### Utilidades reutilizables

Las funciones de carga, separación de variables, preprocesamiento y manejo del desbalance se importan desde `src/modeling_utils.py`.

In [6]:
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [7]:
from src.modeling_utils import (
    build_base_preprocessor,
    get_transformed_feature_names,
    load_split_dataframe,
    split_features_target,
    compute_class_weight_dict,
    build_resampler,
    validate_imbalance_strategy_for_model,
    summarize_class_distribution,
)
print("Utilidades de modelado cargadas correctamente")

Utilidades de modelado cargadas correctamente


## 1. Carga y validación de los datos

Se cargan únicamente las particiones de entrenamiento y validación. El conjunto de prueba permanece reservado para la evaluación final del modelo.

Durante esta primera etapa se utiliza la muestra de desarrollo. No se genera un nuevo particionamiento, sino que se conserva exactamente la asignación `train/valid/test` definida en el notebook 03.

In [8]:
# Columnas necesarias para el modelado
predictor_columns = (preprocessing_config["predictor_candidate_columns"])
target_columns = (preprocessing_config["target_columns"])
metadata_columns = ["clase", "strata_key", "split", "row_id_model"]
modeling_columns = list(dict.fromkeys(predictor_columns + target_columns + metadata_columns))
print("Variables predictoras:", len(predictor_columns))
print("Columnas cargadas en total:", len(modeling_columns))

Variables predictoras: 22
Columnas cargadas en total: 28


In [9]:
# Carga de TRAIN
df_train = load_split_dataframe(
    split_name="train",
    preprocessing_config=preprocessing_config,
    use_dev_sample=USE_DEV_SAMPLE,
    columns=modeling_columns,
    project_root=PROJECT_ROOT,
)
print("TRAIN cargado:", df_train.shape)

TRAIN cargado: (1199179, 28)


In [10]:
# Carga de VALID
df_valid = load_split_dataframe(
    split_name="valid",
    preprocessing_config=preprocessing_config,
    use_dev_sample=USE_DEV_SAMPLE,
    columns=modeling_columns,
    project_root=PROJECT_ROOT,
)
print("VALID cargado:", df_valid.shape)

VALID cargado: (241967, 28)


In [11]:
# Validación estructural
required_columns = set(modeling_columns)
for split_name, df in {"train": df_train, "valid": df_valid}.items():
    missing_columns = (required_columns - set(df.columns))
    if missing_columns:
        raise ValueError(f"{split_name}: faltan columnas: " f"{sorted(missing_columns)}")

    if not df["split"].eq(split_name).all():
        raise ValueError(f"{split_name}: se encontraron registros pertenecientes a otro split.")

    # Los objetivos no deben tener nulos
    if df[preprocessing_config["binary_target_column"]].isna().any():
        raise ValueError(f"{split_name}: attack_a contiene nulos.")

    # Identificador único dentro del split
    if df["row_id_model"].duplicated().any():
        raise ValueError(f"{split_name}: row_id_model contiene duplicados.")

# Validar valores de la tarea binaria
binary_values = set(pd.concat([df_train["attack_a"], df_valid["attack_a"]]).unique())
if not binary_values.issubset({0, 1}):
    raise ValueError("attack_a contiene valores "f"inesperados: {binary_values}")

# Confirmar ausencia de solapamiento entre train y valid
train_ids = (df_train["row_id_model"].to_numpy())
valid_ids = (df_valid["row_id_model"].to_numpy())
overlap_ids = np.intersect1d(train_ids, valid_ids, assume_unique=True)
if overlap_ids.size > 0:
    raise ValueError("Se encontraron registros compartidos entre train y valid.")

print("Validación estructural completada correctamente.")

Validación estructural completada correctamente.


In [12]:
# Distribución binaria
df_binary_distribution = pd.concat(
    [summarize_class_distribution(df_train["attack_a"], name="train"),
        summarize_class_distribution(df_valid["attack_a"], name="valid"),
    ],ignore_index=True,
)
df_binary_distribution

,dataset,clase,n,porcentaje
0,train,0,900000,75.051348
1,train,1,299179,24.948652
2,valid,0,180000,74.390309
3,valid,1,61967,25.609691


In [13]:
# Distribución multiclase solo registros attack_a = 1
y_train_multiclass_check = df_train.loc[df_train["attack_a"].eq(1),"attack_t"]
y_valid_multiclass_check = df_valid.loc[df_valid["attack_a"].eq(1), "attack_t"]
df_multiclass_distribution = pd.concat([
        summarize_class_distribution(y_train_multiclass_check, name="train"),
        summarize_class_distribution(y_valid_multiclass_check, name="valid"),
    ],ignore_index=True,
)
df_multiclass_distribution

,dataset,clase,n,porcentaje
0,train,tcp_red_w,150000,50.137209
1,train,udp_f,65508,21.895922
2,train,icmp_smf,41635,13.916418
3,train,tcp_w32_w,17003,5.683220
4,train,http_f,16071,5.371701
5,train,icmp_f,8139,2.720445
6,train,udp_reaper_w,823,0.275086
7,valid,tcp_red_w,30000,48.412865
8,valid,udp_f,14037,22.652379
9,valid,icmp_smf,8922,14.397986


In [14]:
# Clases de ataque esperadas
ATTACK_CLASS_ORDER = ["icmp_f", "icmp_smf", "udp_f", "http_f", "tcp_w32_w", "tcp_red_w", "udp_reaper_w"]
expected_attack_classes = set(ATTACK_CLASS_ORDER)
for split_name, df in {"train": df_train, "valid": df_valid}.items():
    observed_classes = set(df.loc[df["attack_a"].eq(1), "attack_t"].dropna().unique())
    missing_classes = (expected_attack_classes - observed_classes)
    unexpected_classes = (observed_classes - expected_attack_classes)
    if missing_classes:
        raise ValueError(f"{split_name}: faltan clases de ataque: "f"{sorted(missing_classes)}")
    if unexpected_classes:
        raise ValueError(f"{split_name}: aparecen clases no esperadas: "f"{sorted(unexpected_classes)}")

print("Las siete clases de ataque están presentes en train y valid.")

Las siete clases de ataque están presentes en train y valid.


## 2. Definición de las tareas de clasificación

**Clasificación binaria**

La tarea principal consiste en detectar si un flujo corresponde a tráfico normal o anómalo. Se utiliza `attack_a` como variable objetivo:

- `0`: tráfico normal.
- `1`: tráfico de ataque.

Para esta tarea se utilizan todos los registros de cada partición.

**Clasificación multiclase**

Como tarea complementaria se evalúa la capacidad del modelo para diferenciar entre los tipos de ataque presentes en el conjunto de datos.
Primero se seleccionan únicamente los registros con `attack_a = 1` y posteriormente se utiliza `attack_t` como variable objetivo. Por tanto, el tráfico normal no constituye una clase de esta tarea.
Las clases consideradas son `icmp_f`, `icmp_smf`, `udp_f`, `http_f`, `tcp_w32_w`, `tcp_red_w` y `udp_reaper_w`.

Ambas tareas conservan las particiones definidas en el notebook 03; no se realiza un nuevo split.

In [15]:
# Definición formal de tareas
BINARY_CLASS_ORDER = [0, 1]
ATTACK_CLASS_ORDER = ["icmp_f", "icmp_smf", "udp_f", "http_f", "tcp_w32_w", "tcp_red_w", "udp_reaper_w"]
TASK_CONFIG = {
    "binary": {
        "target_column": preprocessing_config["binary_target_column"],
        "positive_label": 1,
        "class_order": BINARY_CLASS_ORDER,
        "description": "Detección de tráfico normal frente a ataque",
    },
    "multiclass": {
        "target_column":preprocessing_config["multiclass_target_column"],
        "filter_column":preprocessing_config["binary_target_column"],
        "filter_value": 1,
        "class_order": ATTACK_CLASS_ORDER,
        "description": "Clasificación del tipo de ataque",
    },
}
for task_name, config in TASK_CONFIG.items():
    print(f"{task_name.upper()}: " f"{config['description']}")

BINARY: Detección de tráfico normal frente a ataque
MULTICLASS: Clasificación del tipo de ataque


## 3. Construcción del pipeline de Regresión Logística

Se construye un pipeline común que integra el preprocesamiento definido en el notebook 03, el tratamiento opcional del desbalance de clases y el clasificador de Regresión Logística. El flujo general es:

**Datos originales → Preprocesamiento → Estrategia de desbalance opcional → Regresión Logística**

Se consideran cinco escenarios:

1. `none`: distribución de entrenamiento sin modificación.
2. `class_weight`: ajuste de los pesos de las clases sin modificar las observaciones.
3. `smote`: sobremuestreo sintético de las clases minoritarias.
4. `adasyn`: sobremuestreo adaptativo.
5. `smote_enn`: sobremuestreo mediante SMOTE seguido de limpieza mediante ENN.

El preprocesamiento y cualquier técnica de remuestreo serán ajustados exclusivamente durante el entrenamiento. Validación y prueba únicamente recibirán las transformaciones aprendidas.

In [16]:
from sklearn.linear_model import LogisticRegression
from sklearn.exceptions import ConvergenceWarning
from imblearn.pipeline import Pipeline as ImbPipeline
import sklearn
import imblearn
warnings.filterwarnings("default", category=ConvergenceWarning)
print("scikit-learn:", sklearn.__version__)
print("imbalanced-learn:", imblearn.__version__)

scikit-learn: 1.7.1
imbalanced-learn: 0.14.2


### 3.1 Configuración base del clasificador

Antes de realizar optimización de hiperparámetros se establece una configuración inicial común para todos los escenarios de desbalance.
Se utiliza el solver `saga`, adecuado para conjuntos de datos grandes y compatible con clasificación binaria y multiclase. Como configuración inicial se utiliza regularización L2 con `C = 1.0`. Se establece un máximo amplio de iteraciones para reducir el riesgo de detener el entrenamiento antes de alcanzar convergencia. Esta configuración constituye el punto de partida del experimento y posteriormente será sometida a optimización.

In [17]:
# Configuración base de Regresión Logística
LR_BASE_PARAMS = {
    "solver": "saga",
    "penalty": "l2",
    "C": 1.0,
    "fit_intercept": True,
    "max_iter": 1000,
    "tol": 1e-4,
    "random_state": RANDOM_STATE,
}
LR_STRATEGIES = ["none", "class_weight", "smote", "adasyn", "smote_enn"]
print("Parámetros base:")
print(LR_BASE_PARAMS)
print("\nEstrategias:")
print(LR_STRATEGIES)

Parámetros base:
{'solver': 'saga', 'penalty': 'l2', 'C': 1.0, 'fit_intercept': True, 'max_iter': 1000, 'tol': 0.0001, 'random_state': 42}

Estrategias:
['none', 'class_weight', 'smote', 'adasyn', 'smote_enn']


In [18]:
configured_strategies = set(imbalance_config["imbalance_strategies"])
expected_strategies = set(LR_STRATEGIES)
if configured_strategies != expected_strategies:
    raise ValueError("Las estrategias del notebook 04 no coinciden con imbalance_config. "f"Configuradas: {sorted(configured_strategies)} | "
        f"Esperadas: {sorted(expected_strategies)}"
    )
print("Las estrategias de desbalance coinciden con imbalance_config.")

Las estrategias de desbalance coinciden con imbalance_config.


In [19]:
# Etiquetas de entrenamiento
y_train_binary_ref = df_train["attack_a"]
attack_train_mask = df_train["attack_a"].eq(1)
y_train_multiclass_ref = df_train.loc[attack_train_mask, "attack_t"]
print("Tarea binaria:", len(y_train_binary_ref), "registros")
print("Tarea multiclase:", len(y_train_multiclass_ref), "ataques")

Tarea binaria: 1199179 registros
Tarea multiclase: 299179 ataques


In [20]:
binary_class_weights = compute_class_weight_dict(y_train_binary_ref)
multiclass_class_weights = compute_class_weight_dict(y_train_multiclass_ref)
print("Pesos de clase - tarea binaria:")
print(binary_class_weights)
print("\nPesos de clase - tarea multiclase:")
print(multiclass_class_weights)

Pesos de clase - tarea binaria:
{0: 0.6662105555555555, 1: 2.004116264844792}

Pesos de clase - tarea multiclase:
{'http_f': 2.65943980728375, 'icmp_f': 5.251241816298949, 'icmp_smf': 1.0265367393504778, 'tcp_red_w': 0.28493238095238094, 'tcp_w32_w': 2.5136656556406014, 'udp_f': 0.6524372159561754, 'udp_reaper_w': 51.931782676618646}


### 3.2 Constructor del pipeline

Para cada escenario se genera un pipeline independiente. El preprocesador se crea nuevamente en cada ejecución para garantizar que sus estadísticas sean aprendidas exclusivamente durante el ajuste correspondiente.

En el escenario `class_weight` se modifican los pesos utilizados por Regresión Logística, pero no se altera el número de observaciones. En los escenarios SMOTE, ADASYN y SMOTE+ENN se utiliza el remuestreador definido en el notebook 03. Las estrategias de remuestreo no se combinan inicialmente con pesos de clase para poder analizar por separado el efecto de cada tratamiento del desbalance.

In [21]:
def build_logistic_regression_pipeline(task, strategy_name, y_train):
    if task not in {"binary", "multiclass"}:
        raise ValueError("task debe ser 'binary' o 'multiclass'.")
    if strategy_name not in LR_STRATEGIES:
        raise ValueError(f"Estrategia no válida: {strategy_name}")

    validate_imbalance_strategy_for_model(strategy_name=strategy_name, model_family="logistic_regression", imbalance_config=imbalance_config)
    # Preprocesamiento nuevo para cada pipeline
    preprocessor = build_base_preprocessor(preprocessing_config=preprocessing_config, model_family="linear")

    # Pesos solo en el escenario class_weight
    class_weight = (compute_class_weight_dict(y_train)
        if strategy_name == "class_weight"
        else None
    )
    # Remuestreo solo para SMOTE / ADASYN / SMOTE+ENN
    resampler = build_resampler(
        strategy_name=strategy_name,
        task=task,
        y_train=y_train,
        imbalance_config=imbalance_config,
        random_state=RANDOM_STATE,
    )
    classifier = LogisticRegression(**LR_BASE_PARAMS, class_weight=class_weight)
    pipeline = ImbPipeline([
        ("preprocessor", preprocessor),
        ("resampler", resampler if resampler is not None else "passthrough"),
        ("classifier", classifier),
    ])
    return pipeline

**Escenarios experimentales**

| Escenario | Preprocesamiento | Remuestreo | Pesos de clase |
|---|---|---|---|
| `none` | Sí | No | No |
| `class_weight` | Sí | No | Sí |
| `smote` | Sí | SMOTE | No |
| `adasyn` | Sí | ADASYN | No |
| `smote_enn` | Sí | SMOTE + ENN | No |

De esta manera cada escenario modifica únicamente un componente relacionado con el tratamiento del desbalance, manteniendo constante la configuración inicial de Regresión Logística.

In [22]:
lr_binary_pipelines = {
    strategy: build_logistic_regression_pipeline(task="binary", strategy_name=strategy, y_train=y_train_binary_ref)
    for strategy in LR_STRATEGIES
}
print("Pipelines binarios preparados:")
print(list(lr_binary_pipelines))

Pipelines binarios preparados:
['none', 'class_weight', 'smote', 'adasyn', 'smote_enn']


In [23]:
lr_multiclass_pipelines = {
    strategy: build_logistic_regression_pipeline(task="multiclass", strategy_name=strategy, y_train=y_train_multiclass_ref)
    for strategy in LR_STRATEGIES
}
print("Pipelines multiclase preparados:")
print(list(lr_multiclass_pipelines))

Pipelines multiclase preparados:
['none', 'class_weight', 'smote', 'adasyn', 'smote_enn']


In [24]:
def summarize_lr_pipelines(pipelines, task):
    rows = []
    for strategy, pipeline in pipelines.items():
        resampler = pipeline.named_steps["resampler"]
        classifier = pipeline.named_steps["classifier"]
        rows.append({
            "task": task,
            "strategy": strategy,
            "preprocessor": type(pipeline.named_steps["preprocessor"]).__name__,
            "resampler": "None" if resampler == "passthrough" else type(resampler).__name__,
            "class_weight": "Sí" if classifier.class_weight is not None else "No",
            "solver": classifier.solver,
            "C": classifier.C,
            "max_iter": classifier.max_iter,
        })
    return pd.DataFrame(rows)

In [25]:
df_pipeline_summary = pd.concat([
    summarize_lr_pipelines(lr_binary_pipelines, "binary"),
    summarize_lr_pipelines(lr_multiclass_pipelines, "multiclass"),
], ignore_index=True)
df_pipeline_summary

,task,strategy,preprocessor,resampler,class_weight,solver,C,max_iter
0,binary,none,ColumnTransformer,None,No,saga,1.0,1000
1,binary,class_weight,ColumnTransformer,None,Sí,saga,1.0,1000
2,binary,smote,ColumnTransformer,SMOTE,No,saga,1.0,1000
3,binary,adasyn,ColumnTransformer,ADASYN,No,saga,1.0,1000
4,binary,smote_enn,ColumnTransformer,SMOTEENN,No,saga,1.0,1000
5,multiclass,none,ColumnTransformer,None,No,saga,1.0,1000
6,multiclass,class_weight,ColumnTransformer,None,Sí,saga,1.0,1000
7,multiclass,smote,ColumnTransformer,SMOTE,No,saga,1.0,1000
8,multiclass,adasyn,ColumnTransformer,ADASYN,No,saga,1.0,1000
9,multiclass,smote_enn,ColumnTransformer,SMOTEENN,No,saga,1.0,1000


In [26]:
for strategy in ["smote", "adasyn", "smote_enn"]:
    sampler = lr_binary_pipelines[strategy].named_steps["resampler"]
    print(f"{strategy:10s} -> " f"sampling_strategy = {sampler.sampling_strategy}")

smote      -> sampling_strategy = {1: 448768}
adasyn     -> sampling_strategy = {1: 448768}
smote_enn  -> sampling_strategy = {1: 448768}


In [27]:
for strategy in ["smote", "adasyn", "smote_enn"]:
    sampler = lr_multiclass_pipelines[strategy].named_steps["resampler"]
    print(f"\n{strategy}:")
    print(sampler.sampling_strategy)


smote:
{'icmp_smf': 50000, 'tcp_w32_w': 25504, 'http_f': 24106, 'icmp_f': 12208, 'udp_reaper_w': 1234}

adasyn:
{'icmp_smf': 50000, 'tcp_w32_w': 25504, 'http_f': 24106, 'icmp_f': 12208, 'udp_reaper_w': 1234}

smote_enn:
{'icmp_smf': 50000, 'tcp_w32_w': 25504, 'http_f': 24106, 'icmp_f': 12208, 'udp_reaper_w': 1234}


In [28]:
#Comprobación de independencia de pipelines
assert (lr_binary_pipelines["none"].named_steps["preprocessor"] is not lr_binary_pipelines["smote"].named_steps["preprocessor"])
assert (lr_binary_pipelines["none"].named_steps["classifier"] is not lr_binary_pipelines["class_weight"].named_steps["classifier"])
print("Cada escenario contiene instancias independientes.")

Cada escenario contiene instancias independientes.


## 4. Funciones de evaluación

Se definen funciones comunes para evaluar de manera uniforme todos los escenarios de Regresión Logística. Estas funciones se utilizarán posteriormente sobre validación y prueba sin modificar el modelo ni los datos evaluados.

Para la **clasificación binaria**, la clase positiva corresponde al tráfico de ataque (`attack_a = 1`). Se reportan Precision, Recall, F1, PR-AUC, ROC-AUC, Balanced Accuracy y matriz de confusión.

Para la **clasificación multiclase**, Macro-F1 constituye la métrica principal debido al desbalance entre tipos de ataque. También se reportan Weighted-F1, Precision y Recall macro, Balanced Accuracy, métricas individuales por clase y matriz de confusión.

In [29]:
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, balanced_accuracy_score,
    precision_recall_curve, auc,
    confusion_matrix, precision_recall_fscore_support
)

### 4.1 Evaluación de la tarea binaria

La evaluación binaria utiliza la probabilidad estimada para la clase ataque. Inicialmente se utiliza un umbral de decisión de `0.50`.
Además de las métricas agregadas, se almacenan los valores TN, FP, FN y TP para facilitar el análisis posterior de falsas alarmas y ataques no detectados.

In [30]:
def evaluate_binary_predictions(y_true, y_score, threshold=0.50, positive_label=1):
    """Evalúa la clasificación binaria a partir de probabilidades de ataque."""
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)
    y_pred = (y_score >= threshold).astype(np.int8)

    precision = precision_score(y_true, y_pred, pos_label=positive_label, zero_division=0)
    recall = recall_score(y_true, y_pred, pos_label=positive_label, zero_division=0)
    f1 = f1_score(y_true, y_pred, pos_label=positive_label, zero_division=0)
    roc_auc = roc_auc_score(y_true, y_score)

    precision_curve, recall_curve, _ = precision_recall_curve(y_true, y_score, pos_label=positive_label)
    pr_auc = auc(recall_curve, precision_curve)

    balanced_acc = balanced_accuracy_score(y_true, y_pred)

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    metrics = pd.DataFrame([{
        "threshold": threshold,
        "precision_attack": precision,
        "recall_attack": recall,
        "f1_attack": f1,
        "pr_auc": pr_auc,
        "roc_auc": roc_auc,
        "balanced_accuracy": balanced_acc,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "n": len(y_true),
    }])

    cm_df = pd.DataFrame(cm, index=["real_normal", "real_attack"], columns=["pred_normal", "pred_attack"])
    return metrics, cm_df, y_pred

In [31]:
def get_binary_attack_scores(fitted_pipeline, X, positive_label=1):
    """Obtiene P(ataque) a partir de un pipeline ya entrenado."""
    probabilities = fitted_pipeline.predict_proba(X)
    classes = fitted_pipeline.named_steps["classifier"].classes_
    positive_idx = np.where(classes == positive_label)[0]
    if len(positive_idx) != 1:
        raise ValueError(f"No se encontró correctamente la clase positiva {positive_label}.")

    return probabilities[:, positive_idx[0]]

### 4.2 Evaluación de la tarea multiclase

En la clasificación del tipo de ataque se utiliza Macro-F1 como métrica principal, ya que otorga el mismo peso a cada clase independientemente de su frecuencia. Se calculan también Weighted-F1, Precision y Recall macro, Balanced Accuracy y las métricas individuales de cada ataque.

In [32]:
def evaluate_multiclass_predictions(y_true, y_pred, class_order):
    """Evalúa la clasificación multiclase y genera métricas globales y por clase."""
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    macro_precision = precision_score(y_true, y_pred, average="macro", zero_division=0)
    macro_recall = recall_score(y_true, y_pred, average="macro", zero_division=0)
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    weighted_f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    balanced_acc = balanced_accuracy_score(y_true, y_pred)

    summary = pd.DataFrame([{
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "balanced_accuracy": balanced_acc,
        "n": len(y_true),
    }])
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=class_order, zero_division=0
    )
    per_class = pd.DataFrame({
        "class": class_order,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "support": support.astype(int),
    })
    cm = confusion_matrix(y_true, y_pred, labels=class_order)
    cm_df = pd.DataFrame(cm, index=[f"real_{label}" for label in class_order], columns=[f"pred_{label}" for label in class_order])
    return summary, per_class, cm_df

In [33]:
PRIMARY_METRIC = {
    "binary": "pr_auc",
    "multiclass": "macro_f1",
}
SECONDARY_METRICS = {
    "binary": ["precision_attack", "recall_attack", "f1_attack", "roc_auc", "balanced_accuracy"],
    "multiclass": ["macro_precision", "macro_recall", "weighted_f1", "balanced_accuracy"],
}
print("Métrica principal binaria:", PRIMARY_METRIC["binary"])
print("Métrica principal multiclase:", PRIMARY_METRIC["multiclass"])

Métrica principal binaria: pr_auc
Métrica principal multiclase: macro_f1


**Función para añadir información del experimento**

In [34]:
def add_experiment_metadata(metrics_df, task, strategy, split, dataset_mode=DATASET_MODE):
    """Añade información común para identificar cada resultado experimental."""
    result = metrics_df.copy()
    result.insert(0, "dataset_mode", dataset_mode)
    result.insert(1, "model", "logistic_regression")
    result.insert(2, "task", task)
    result.insert(3, "strategy", strategy)
    result.insert(4, "split", split)

    return result

In [35]:
binary_experiment_results = []
multiclass_experiment_results = []
print("Estructuras de resultados inicializadas.")

Estructuras de resultados inicializadas.


## 5. Experimento binario base

Se entrena una primera Regresión Logística para establecer el desempeño de referencia de la tarea binaria. El modelo se ajusta exclusivamente con `train` y se evalúa sobre `valid`. Este escenario utiliza:

- La configuración base de Regresión Logística definida anteriormente.
- La distribución original de la muestra de entrenamiento.
- Sin pesos de clase.
- Sin SMOTE, ADASYN ni SMOTE+ENN.
- Umbral inicial de clasificación de `0.50`.



In [36]:
# Datos para la tarea binaria
X_train_binary, y_train_binary = split_features_target(df_train, preprocessing_config=preprocessing_config, task="binary")
X_valid_binary, y_valid_binary = split_features_target(df_valid, preprocessing_config=preprocessing_config, task="binary")
print("TRAIN:", X_train_binary.shape, y_train_binary.shape)
print("VALID:", X_valid_binary.shape, y_valid_binary.shape)
display(
    pd.concat([
        summarize_class_distribution(y_train_binary, name="train"),
        summarize_class_distribution(y_valid_binary, name="valid"),
    ], ignore_index=True)
)

TRAIN: (1199179, 22) (1199179,)
VALID: (241967, 22) (241967,)


,dataset,clase,n,porcentaje
0,train,0,900000,75.051348
1,train,1,299179,24.948652
2,valid,0,180000,74.390309
3,valid,1,61967,25.609691


In [37]:
# Pipeline baseline
BASELINE_STRATEGY = "none"
lr_binary_baseline = lr_binary_pipelines[BASELINE_STRATEGY]
lr_binary_baseline

,steps,"[('preprocessor', ...), ('resampler', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num_log', ...), ('flags', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


### 5.1 Entrenamiento del baseline

El preprocesador y la Regresión Logística se ajustan conjuntamente utilizando únicamente el conjunto de entrenamiento. Las estadísticas utilizadas para escalamiento, codificación por frecuencia, codificación de IP y demás transformaciones se aprenden exclusivamente durante este `fit`.

In [38]:
import time
start_time = time.perf_counter()
lr_binary_baseline.fit(X_train_binary, y_train_binary)
baseline_fit_time = time.perf_counter() - start_time
print(f"Entrenamiento completado en {baseline_fit_time / 60:.2f} minutos.")

Entrenamiento completado en 0.83 minutos.


In [ ]:
#Verificación de convergencia
baseline_classifier = lr_binary_baseline.named_steps["classifier"]
print("Iteraciones utilizadas:", baseline_classifier.n_iter_)
print("Máximo permitido:", baseline_classifier.max_iter)
if np.max(baseline_classifier.n_iter_) >= baseline_classifier.max_iter:
    print("ADVERTENCIA: el modelo alcanzó max_iter; conviene revisar convergencia.")
else:
    print("El modelo terminó antes de alcanzar max_iter.")

Iteraciones utilizadas: [19]
Máximo permitido: 1000
El modelo terminó antes de alcanzar max_iter.


**Variables generadas por el preprocesamiento**

In [40]:
baseline_preprocessor = lr_binary_baseline.named_steps["preprocessor"]
baseline_feature_names = get_transformed_feature_names(baseline_preprocessor)
if baseline_feature_names is not None:
    print("Número de variables después del preprocesamiento:", len(baseline_feature_names))
    print("\nPrimeras variables:")
    print(baseline_feature_names[:20])

Número de variables después del preprocesamiento: 49

Primeras variables:
['num_log__td' 'num_log__ipkt' 'num_log__ibyt' 'flags__tcp_flag_fin'
 'flags__tcp_flag_syn' 'flags__tcp_flag_rst' 'flags__tcp_flag_psh'
 'flags__tcp_flag_ack' 'flags__tcp_flag_urg' 'flags__tcp_flag_ece'
 'flags__tcp_flag_cwr' 'onehot__pr_AH' 'onehot__pr_ESP' 'onehot__pr_GRE'
 'onehot__pr_ICMP' 'onehot__pr_ICMP6' 'onehot__pr_IGMP' 'onehot__pr_IPIP'
 'onehot__pr_IPv6' 'onehot__pr_OSPF']


### 5.2 Evaluación sobre validación

El pipeline entrenado se aplica al conjunto de validación sin reajustar ninguna transformación. Se obtiene la probabilidad estimada de pertenecer a la clase ataque y, como punto de referencia inicial, se utiliza un umbral de `0.50`.

In [41]:
start_time = time.perf_counter()
y_valid_score_baseline = get_binary_attack_scores(lr_binary_baseline, X_valid_binary, positive_label=1)
baseline_predict_time = time.perf_counter() - start_time
print(f"Predicciones generadas en {baseline_predict_time:.2f} segundos.")
print("Rango de probabilidades:", float(y_valid_score_baseline.min()), "-", float(y_valid_score_baseline.max()))

Predicciones generadas en 2.93 segundos.
Rango de probabilidades: 7.053303253956589e-13 - 0.9988356232643127


In [42]:
#Métricas del baseline
baseline_metrics, baseline_cm, y_valid_pred_baseline = evaluate_binary_predictions(
    y_true=y_valid_binary,
    y_score=y_valid_score_baseline,
    threshold=0.50,
    positive_label=1,
)
baseline_metrics["fit_time_seconds"] = baseline_fit_time
baseline_metrics["predict_time_seconds"] = baseline_predict_time
baseline_metrics = add_experiment_metadata(baseline_metrics, task="binary", strategy=BASELINE_STRATEGY, split="valid")
baseline_metrics

,dataset_mode,model,task,strategy,split,threshold,precision_attack,recall_attack,f1_attack,pr_auc,roc_auc,balanced_accuracy,tn,fp,fn,tp,n,fit_time_seconds,predict_time_seconds
0,development,logistic_regression,binary,none,valid,0.5,0.94094,0.863347,0.900475,0.958989,0.986648,0.922346,176642,3358,8468,53499,241967,49.622765,2.928357


In [43]:
baseline_cm

,pred_normal,pred_attack
real_normal,176642,3358
real_attack,8468,53499


In [44]:
# Cálculo de FPR y FNR
tn = baseline_metrics.loc[0, "tn"]
fp = baseline_metrics.loc[0, "fp"]
fn = baseline_metrics.loc[0, "fn"]
tp = baseline_metrics.loc[0, "tp"]

fpr = fp / (fp + tn)
fnr = fn / (fn + tp)

print(f"False Positive Rate (FPR): {fpr:.6f}")
print(f"False Negative Rate (FNR): {fnr:.6f}")

False Positive Rate (FPR): 0.018656
False Negative Rate (FNR): 0.136653


In [45]:
binary_experiment_results.append(baseline_metrics.copy())
df_binary_experiments = pd.concat(binary_experiment_results, ignore_index=True)
df_binary_experiments

,dataset_mode,model,task,strategy,split,threshold,precision_attack,recall_attack,f1_attack,pr_auc,roc_auc,balanced_accuracy,tn,fp,fn,tp,n,fit_time_seconds,predict_time_seconds
0,development,logistic_regression,binary,none,valid,0.5,0.94094,0.863347,0.900475,0.958989,0.986648,0.922346,176642,3358,8468,53499,241967,49.622765,2.928357


In [46]:
BASELINE_RESULTS_PATH = LR_BINARY_RESULTS_PATH / "baseline_none"
BASELINE_RESULTS_PATH.mkdir(parents=True, exist_ok=True)
baseline_metrics.to_csv(BASELINE_RESULTS_PATH / "validation_metrics.csv", index=False, encoding="utf-8-sig")
baseline_cm.to_csv(BASELINE_RESULTS_PATH / "confusion_matrix.csv", encoding="utf-8-sig")
print("Resultados guardados en:")
print(BASELINE_RESULTS_PATH)

Resultados guardados en:
C:\Users\Laura\Documents\TrabajoGrado2026\TG2026\results\modeling\logistic_regression\binary\baseline_none


In [47]:
baseline_summary = baseline_metrics.iloc[0]
print("BASELINE BINARIO — VALIDACIÓN")
print("-" * 40)
print(f"PR-AUC:            {baseline_summary['pr_auc']:.4f}")
print(f"ROC-AUC:           {baseline_summary['roc_auc']:.4f}")
print(f"Precision ataque:  {baseline_summary['precision_attack']:.4f}")
print(f"Recall ataque:     {baseline_summary['recall_attack']:.4f}")
print(f"F1 ataque:         {baseline_summary['f1_attack']:.4f}")
print(f"Balanced accuracy: {baseline_summary['balanced_accuracy']:.4f}")
print()
print(f"FP: {int(baseline_summary['fp']):,}")
print(f"FN: {int(baseline_summary['fn']):,}")

BASELINE BINARIO — VALIDACIÓN
----------------------------------------
PR-AUC:            0.9590
ROC-AUC:           0.9866
Precision ataque:  0.9409
Recall ataque:     0.8633
F1 ataque:         0.9005
Balanced accuracy: 0.9223

FP: 3,358
FN: 8,468
